In [45]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timezone
import requests
import os
from dotenv import load_dotenv
load_dotenv()

True

Get Top Collections

In [46]:
api_key = os.getenv('opensea_api')
url = "https://api.opensea.io/api/v2/collections/top?sort_by=one_day_volume&chains=ethereum&limit=100"
headers = { "accept": "*/*", "x-api-key": api_key}
response = requests.get(url, headers=headers)
collection = response.json()
collection_temp = collection.get("collections", [])
collection_slugs = [s for s in (item.get('collection') for item in collection_temp) if s]

In [47]:
result = []
for slug in collection_slugs:
    stats_url = f'https://api.opensea.io/api/v2/collections/{slug}/stats'
    price_url = f'https://api.opensea.io/api/v2/collections/{slug}/floor_prices?timeframe=one_year'
    headers = {"accept": "*/*", "x-api-key": api_key}
    
    try:
        stat_response = requests.get(stats_url, headers=headers, timeout=10)
        price_response = requests.get(price_url, headers=headers, timeout=10)
        stat_response.raise_for_status()
        price_response.raise_for_status()
    except requests.RequestException as e:
        print(f"Failed to fetch data for {slug}: {e}")
        continue

    data = stat_response.json()
    price = price_response.json()
    total = data.get('total', {})
    intervals = {item.get('interval'): item for item in data.get('intervals', [])}

    prices = price.get('floor_prices', [])
    floor_price_1y = min(prices, key=lambda x: x['time']).get('token_unit') if prices else None

    result.append({
        'slug': slug,
        'volume_1d': intervals.get('one_day', {}).get('volume'),
        'volume_7d': intervals.get('seven_day', {}).get('volume'),
        'volume_30d': intervals.get('thirty_day', {}).get('volume'),
        'floor_price': total.get('floor_price'),
        'floor_price_1y_ago': floor_price_1y,
        'num_owners': total.get('num_owners'),
    })
        
df= pd.DataFrame(result)
df['avg_vol_per_owner_30d'] = df['volume_30d'] / df['num_owners'].replace(0, np.nan)
df['volume_share_week'] = df['volume_1d'] / df['volume_7d'].replace(0, np.nan) #to compare 1day volume to total 7day volume
df['volume_share_month'] = df['volume_1d'] / df['volume_30d'].replace(0, np.nan) #to compare 1day volume to total 30day volume
df['volume_momentum_7d'] = df['volume_1d'] / (df['volume_7d'] / 7).replace(0, np.nan) #to compare 1day volume to daily average in the last 7 days
df['volume_momentum_30d'] = df['volume_1d'] / (df['volume_30d'] / 30).replace(0, np.nan) #to compare 1day volume to daily average in the last 30 days
df['floor_return_1y'] = (df['floor_price'] - df['floor_price_1y_ago']) / df['floor_price_1y_ago'] #to determine price return
df['collected_at'] = datetime.now(timezone.utc).isoformat()
print(df.head(2))

                slug   volume_1d   volume_7d   volume_30d  floor_price  \
0      pudgypenguins  110.580389  559.056830  2240.128524     4.799999   
1  boredapeyachtclub  109.619790  634.187392  3663.819864     9.199000   

   floor_price_1y_ago  num_owners  avg_vol_per_owner_30d  volume_share_week  \
0                9.00        5089               0.440190           0.197798   
1               11.11        5619               0.652041           0.172851   

   volume_share_month  volume_momentum_7d  volume_momentum_30d  \
0            0.049363            1.384587             1.480902   
1            0.029920            1.209956             0.897586   

   floor_return_1y                      collected_at  
0        -0.466667  2026-06-21T22:50:20.519190+00:00  
1        -0.172007  2026-06-21T22:50:20.519190+00:00  


In [48]:
import sqlite3
conn = sqlite3.connect("nft_data.db")

df.to_sql(
    "collection_stats",
    conn,
    if_exists="append",
    index=False
)

conn.close()